
# S&P 500 Survivorship-Bias-Free & Point-in-Time (PIT) Dataset

Welcome to the official companion notebook for the **S&P 500 Quantitative Dataset** project! This repository programmatically builds a professional-grade, **Survivorship-Bias-Free** and **Point-in-Time (PIT)** daily financial dataset from scratch.

## Why Local Execution is Required (Repo & Cloning Guide)

To comply with the strict terms of service of third-party financial data providers (which prohibit the unauthorized bulk redistribution of raw proprietary financial data), **the complete dataset cannot be directly pre-packaged or hosted inside a static Kaggle environment**.

To generate the full, production-ready dataset containing advanced Point-in-Time merges, SEC EDGAR fundamentals, and Tiingo price backfilling, **you must clone the repository and run the pipeline locally or via Docker**.

### How to Clone and Run the Pipeline

1. **Clone the repository:**

```bash
git clone https://github.com/K0D1Z/sp500-quantitative-dataset.git
cd sp500-quantitative-dataset

```


2. **Configure your environment variables:**


Create a `.env` file in the root directory and supply your free Tiingo API key (required for historical delisted price backfilling):


```env
TIINGO_API_KEY=your_actual_tiingo_api_key_here

```


3. **Execute the master pipeline (`main.py`):**

* **Via Docker Compose:**

```bash
docker compose up --build

```


* **Via Local Python (`uv`):**

```bash
uv sync
export TIINGO_API_KEY="your_actual_tiingo_api_key_here"
uv run python main.py

```


## Complete Pipeline Architecture (`main.py`)

The master entry point **`main.py`** orchestrates a rate-limit resilient, multi-stage ETL pipeline designed to eliminate look-ahead bias and structural data distortion:

1. **Daily Composition Generation (`generate_daily_composition.py`):** Re-engineers the daily index history from 2015 onwards by working backwards from current constituents and historical change logs, capturing delisted, acquired, and bankrupt entities (e.g., SVB, First Republic Bank).


2. **Corporate Events Ledger (`generate_corporate_events.py`):** Scrapes and categorizes structural index removals into standardized tags (`ACQUISITION`, `MARKET_CAP`, `SPIN_OFF`, `BANKRUPTCY`, `OTHER`).


3. **Historical Price Downloading & Backfilling (`download_historical_prices.py` & `backfill_missing_prices.py`):** Pulls daily OHLCV data using Yahoo Finance (`yfinance`) and seamlessly backfills missing or delisted series via the Tiingo API using a stateful batching system.


4. **SEC EDGAR Fundamentals Extraction (`fetch_sec_fundamentals.py`):** Parses raw US-GAAP XBRL financial facts (Income Statements, Balance Sheets, Cash Flows) directly from the SEC EDGAR database.


5. **Point-in-Time (PIT) Merge & Feature Engineering (`feature_engineering.py`):** Executes strict `pd.merge_asof` matching based on official SEC **Filing Dates** (eliminating look-ahead bias), computes Trailing Twelve Months (TTM) normalized metrics, backward cumulative split adjustments, valuation multiples (P/E, P/B, Market Cap), and technical indicators (RSI, Volatility, SMAs).


## Final Dataset Structure & Columns

The generated production-ready dataset (`data/datasets/daily_features/s_and_p_500_daily_features.parquet` / `.csv`) contains over **1.6+ million rows** and **45+ features** divided into four main categories:

### 1. Market & Price Features (OHLCV)



* **`Date`**: Trading session date (`YYYY-MM-DD`).


* **`Ticker`**: Equity ticker symbol (e.g., `AAPL`, `BRK.B`).


* **`Open`**, **`High`**, **`Low`**, **`Close`**: Unadjusted daily price metrics ($).


* **`Adj Close`**: Split and dividend-adjusted closing price ($).


* **`Volume`**: Total number of shares traded during the session.



### 2. Corporate Actions & Split Adjustments



* **`Stock Splits`** & **`Split Multiplier`**: Raw split execution ratios (`1.0` if none).


* **`Split Shifted`**: Multiplier shifted backwards by 1 day to handle execution timing.


* **`Cum Split Factor`**: Time-reversibly adjusted cumulative product of historical splits.



### 3. Point-in-Time SEC EDGAR Fundamentals (TTM Aggregated)



* **`Filing Date`** & **`Period End`**: Official SEC public release date and fiscal period end.


* **`Form`**: SEC filing identifier (`10-K` or `10-Q`).


* **Core Financial Metrics (TTM)**: `Revenue`, `Cost of Revenue`, `Gross Profit`, `Operating Expenses`, `R&D Expenses`, `SG&A Expenses`, `Operating Income`, `Net Income`, `EPS (Basic)`, `EPS (Diluted)`.


* **Balance Sheet Items**: `Total Assets`, `Current Assets`, `Cash and Equivalents`, `Total Liabilities`, `Current Liabilities`, `Long-Term Debt`, `Stockholders Equity`.


* **Cash Flow & Shares**: `Operating Cash Flow`, `CapEx`, `Dividends Paid`, `Stock Repurchases`, `Shares Outstanding (Basic/Diluted)`.



### 4. Valuation Multiples & Technical Indicators



* **Valuation**: `Market Cap`, `P/E Ratio` (using TTM Diluted EPS), `P/B Ratio`.


* **Technicals**: `SMA_50`, `SMA_200`, `Volatility_30D` (30-day rolling return std dev), `RSI_14`.


* **Metadata**: `GICS Sector`, `GICS Sub-Industry`, `CIK` (10-digit zero-padded SEC identifier).